<a href="https://colab.research.google.com/github/shivangi221b/ASR-Fine-Tuning-Modules-and-RL-Adaptation-Analysis/blob/shivangi_initial_setup/STT_Domain_Adaptation_NeMo_Practical_Validation_Script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STT Domain Adaptation - NeMo Practical Validation Script
Goal:
  1. Baseline fine-tuning using NVIDIA NeMo on Common Voice 17.0 (1000 samples)
  2. Document NeMo training loop behaviour + hook points
  3. Test all three custom loss/reward injections:
       - MWER  (WER-based policy gradient)
       - WWER  (Weighted WER - upweights domain terms)
       - LLM   (Mock LLM scorer placeholder)

Model  : stt_en_conformer_ctc_small (~13M params, fastest NeMo ASR model)
         Upgrade to stt_en_conformer_ctc_medium for better WER on GCP
Runtime: Colab Pro A100 — smoke test (200 samples, 1 epoch) ~20 min
                        — full run (1000 samples, 3 epochs) ~50 min
         GCP A100       — full run ~25 min

CHANGES FROM HF VERSION:
  - Install: NeMo toolkit instead of transformers/accelerate
  - Model: NeMo EncDecCTCModelBPE (Conformer-CTC) instead of Whisper
  - Data: NeMo manifest JSON format (converter included below)
  - Training: PyTorch Lightning via NeMo trainer instead of HF Seq2SeqTrainer
  - Loss hook: _compute_loss() override on model class (cleaner than HF)
  - Reward functions: UNCHANGED (MWER, WWER, LLM mock identical)
  - Dataset: Common Voice 17.0 KEPT — manifest converter handles format
  - Domain terms, weights, reward tests: UNCHANGED

NOTE ON COMMON VOICE 17.0:
  As of October 2025, CV17 requires Mozilla Data Collective login.
  The dataset loader below handles this. If login fails, swap
  load_common_voice() for load_librispeech() provided at the bottom.

# CELL 1 — Install dependencies

In [4]:
pip install -q nemo_toolkit[asr]

In [3]:
!pip install -q datasets soundfile librosa jiwer

# CELL 2 — Imports & Config

In [40]:
import os
import json
import time
import logging
import tempfile
from pathlib import Path
from typing import Dict, List, Optional

import torch
import numpy as np
import soundfile as sf
from omegaconf import OmegaConf, DictConfig

# NeMo imports
import nemo
import nemo.collections.asr as nemo_asr
from nemo.collections.asr.models import EncDecCTCModelBPE
from nemo.core.config import hydra_runner
from nemo.utils import logging as nemo_logging
from nemo.utils.exp_manager import exp_manager

# PyTorch Lightning (NeMo's training backbone)
import lightning.pytorch as pl
from lightning.pytorch.callbacks import Callback

# Data utilities
from datasets import load_dataset, Audio

# WER (kept from HF version — used by reward functions)
from jiwer import wer as compute_wer_jiwer

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

In [48]:
# ---- CONFIG (edit these) ----
NEMO_MODEL_NAME  = "stt_en_conformer_ctc_small"   # ~13M params, fastest
                                                    # swap to stt_en_conformer_ctc_medium for better WER
TRAIN_SAMPLES    = 200    # START HERE for smoke test on Colab
                          # increase to 1000 for full run
EVAL_SAMPLES     = 100    # smoke: 100, full: 200
BATCH_SIZE       = 8     # A100: 16; T4: 8, OG = 16
LEARNING_RATE    = 1e-4   # NeMo default for CTC fine-tuning (higher than Whisper)
NUM_EPOCHS       = 1      # smoke: 1, full: 3
OUTPUT_DIR       = "./nemo-cv17-baseline"
MANIFEST_DIR     = "./manifests"
REWARD_MODE      = "mwer"  # "mwer" | "wwer" | "llm" | "all"

# Domain-specific terms for WWER — UNCHANGED from HF version
DOMAIN_TERMS = {
    # healthcare
    "myocardial", "infarction", "hypertension", "tachycardia",
    "arrhythmia", "stethoscope", "auscultation", "echocardiogram",
    # judicial
    "plaintiff", "defendant", "affidavit", "subpoena",
    "jurisdiction", "deposition", "mandamus", "injunction",
}
DOMAIN_TERM_WEIGHT = 3.0
REWARD_WEIGHT = 0.3

# CELL 3 — Load Dataset & Convert to NeMo Manifest Format
 NeMo does NOT use HuggingFace DataLoader directly.
 It requires a manifest: a .json file where each line is:
   {"audio_filepath": "/path/to/audio.wav", "text": "transcription", "duration": 3.2}

 This cell:
   1. Loads Common Voice 17.0 via HuggingFace datasets
   2. Saves audio as .wav files
   3. Writes NeMo manifest JSON

In [3]:
def load_common_voice(train_n: int, eval_n: int):
    """
    Load Common Voice 17.0 from HuggingFace.
    Requires: huggingface-cli login + CV17 terms accepted at
    https://huggingface.co/datasets/mozilla-foundation/common_voice_17_0
    """
    logger.info("Loading Common Voice 17.0 ...")
    ds = load_dataset(
        "mozilla-foundation/common_voice_17_0",
        "en",
        split={
            "train":      f"train[:{train_n}]",
            "validation": f"validation[:{eval_n}]",
        },
        trust_remote_code=True,
        token=True,
    )
    ds = ds.cast_column("audio", Audio(sampling_rate=16_000))
    logger.info(f"  Train: {len(ds['train'])} | Eval: {len(ds['validation'])}")
    return ds

In [4]:
def load_librispeech_fallback(train_n: int, eval_n: int):
    """
    Fallback if CV17 login fails. LibriSpeech needs no login.
    NOTE: text field is 'text' not 'sentence' — handled in build_manifest().
    """
    logger.info("Loading LibriSpeech (fallback) ...")
    ds = load_dataset(
        "librispeech_asr", "clean",
        split={
            "train":      f"train.100[:{train_n}]",
            "validation": f"validation[:{eval_n}]",
        },
        trust_remote_code=True,
    )
    ds = ds.cast_column("audio", Audio(sampling_rate=16_000))
    return ds, "text"   # returns text field name too

In [5]:
def build_nemo_manifest(dataset, split_name: str, audio_dir: str,
                        text_field: str = "sentence") -> str:
    """
    Convert HuggingFace dataset split → NeMo manifest JSON.

    Saves .wav files to audio_dir and writes manifest to MANIFEST_DIR.
    Returns path to manifest file.

    WHAT THIS DOES (important for paper documentation):
      NeMo's data pipeline expects pre-saved .wav files + a manifest.
      This is different from HF Trainer which streams audio on-the-fly.
      Hook point: manifest is loaded by NeMo's AudioToCharDataset.
    """
    os.makedirs(audio_dir, exist_ok=True)
    os.makedirs(MANIFEST_DIR, exist_ok=True)

    manifest_path = os.path.join(MANIFEST_DIR, f"{split_name}_manifest.json")
    written = 0

    with open(manifest_path, "w") as f:
        for i, sample in enumerate(dataset):
            audio_array = sample["audio"]["array"]
            sr          = sample["audio"]["sampling_rate"]
            text        = sample[text_field].strip().lower()  # NeMo expects lowercase

            if not text:
                continue  # skip empty transcriptions

            # Save audio as .wav
            wav_path = os.path.join(audio_dir, f"{split_name}_{i:05d}.wav")
            sf.write(wav_path, audio_array, sr)

            duration = len(audio_array) / sr

            # Write manifest entry (one JSON per line)
            entry = {
                "audio_filepath": os.path.abspath(wav_path),
                "text":           text,
                "duration":       round(duration, 3),
            }
            f.write(json.dumps(entry) + "\n")
            written += 1

    logger.info(f"  Manifest written: {manifest_path} ({written} samples)")
    return manifest_path

# CELL 4 — NeMo Model Loading
 NeMo handles all feature extraction internally (no separate Processor).
 The model includes: FilterbankFeatureExtractor → Conformer encoder → CTC decoder
 This is the key architectural difference from Whisper (encoder-decoder).


In [29]:
def load_nemo_model(model_name: str) -> EncDecCTCModelBPE:
    """
    Load pretrained NeMo ASR model from NVIDIA NGC.
    Model is downloaded automatically on first run (~500MB for small).

    stt_en_conformer_ctc_small  : ~13M params  — use for Colab smoke test
    stt_en_conformer_ctc_medium : ~30M params  — use for GCP full run
    stt_en_fastconformer_ctc_large: ~120M params — use for production

    HOOK POINT NOTE:
      The model's _compute_loss() method is where we inject reward-based loss.
      See Cell 8 (NemoRewardModel) for the override.
    """
    logger.info(f"Loading NeMo model: {model_name}")
    model = nemo_asr.models.EncDecCTCModelBPE.from_pretrained(model_name)
    logger.info(f"  Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")
    return model

# CELL 5 — NeMo Data Config
NeMo uses OmegaConf/Hydra for config — different from HF TrainingArguments.
 This builds the data config programmatically (no YAML file needed).

In [7]:
def build_data_config(train_manifest: str, val_manifest: str) -> DictConfig:
    """
    Build NeMo data configuration.
    This replaces the HF DataCollator + DataLoader setup.
    """
    cfg = OmegaConf.create({
        "train_ds": {
            "manifest_filepath": train_manifest,
            "sample_rate":       16000,
            "batch_size":        BATCH_SIZE,
            "trim_silence":      True,
            "shuffle":           True,
            "num_workers":       2,
            "pin_memory":        True,
        },
        "validation_ds": {
            "manifest_filepath": val_manifest,
            "sample_rate":       16000,
            "batch_size":        BATCH_SIZE,
            "shuffle":           False,
            "num_workers":       2,
            "pin_memory":        True,
        },
    })
    return cfg

# CELL 6 — WER Evaluation
 NeMo has built-in WER tracking during training.
 We also keep standalone WER for reward functions.

In [43]:
def evaluate_wer(model, manifest_path: str) -> float:
    """
    Evaluate WER on a manifest file.
    Returns WER as percentage (0-100).
    """
    # Load references from manifest
    references = []
    audio_paths = []
    with open(manifest_path, "r") as f:
        for line in f:
            entry = json.loads(line)
            references.append(entry["text"])
            audio_paths.append(entry["audio_filepath"])

    # Transcribe with NeMo
    hypotheses = model.transcribe(audio_paths, batch_size=BATCH_SIZE)

    # Handle NeMo's return format - can be list of Hypothesis objects or list of strings
    # NeMo 2.x returns Hypothesis objects with .text attribute
    if hypotheses and hasattr(hypotheses[0], 'text'):
        hypotheses = [h.text for h in hypotheses]
    elif hypotheses and isinstance(hypotheses[0], list):
        # Some models return list of lists (beam search)
        hypotheses = [h[0] if isinstance(h[0], str) else h[0].text for h in hypotheses]

    # Ensure all hypotheses are strings
    hypotheses = [str(h) if not isinstance(h, str) else h for h in hypotheses]

    wer = compute_wer_jiwer(references, hypotheses) * 100
    logger.info(f"  Evaluation WER: {wer:.2f}%")
    return wer

 # CELL 7 — HOOK POINTS (where RL reward injection happens)
 These are the three "hook points" documented for the paper.
In a real RL training loop you would:
   1. Forward pass → get logits
   2. Sample hypotheses from logits (not just argmax)
   3. Score each hypothesis with one of the reward functions below
   4. Weight the cross-entropy loss by (reward - baseline)
   5. Backward pass on weighted loss

 These are identical to the HF script.
 The hook point where they're called changes (see Cell 8),
 but the functions themselves are the same.

In [9]:
# ---- REWARD FUNCTION 1: MWER ----

def compute_mwer_reward(hypotheses: List[str], references: List[str]) -> torch.Tensor:
    """
    MWER reward: reward = 1 - WER
    UNCHANGED from HF version.
    Hook point: inside NemoRewardModel._compute_loss() — see Cell 8.
    """
    rewards = []
    for hyp, ref in zip(hypotheses, references):
        if not ref.strip():
            rewards.append(0.5)
            continue
        error_rate = compute_wer_jiwer(ref, hyp)
        reward = max(0.0, 1.0 - error_rate)
        rewards.append(reward)
    return torch.tensor(rewards, dtype=torch.float32)

In [10]:
# ---- REWARD FUNCTION 2: WWER ----

def compute_wwer_reward(
    hypotheses: List[str],
    references: List[str],
    domain_terms: set = DOMAIN_TERMS,
    term_weight: float = DOMAIN_TERM_WEIGHT,
) -> torch.Tensor:
    """
    WWER: domain term errors penalised more heavily.
    UNCHANGED from HF version.
    """
    rewards = []
    for hyp, ref in zip(hypotheses, references):
        ref_words = ref.lower().split()
        hyp_words = hyp.lower().split()

        if not ref_words:
            rewards.append(0.5)
            continue

        weights    = [term_weight if w in domain_terms else 1.0 for w in ref_words]
        total_w    = sum(weights)
        errors     = 0.0
        min_len    = min(len(ref_words), len(hyp_words))

        for i in range(min_len):
            if ref_words[i] != hyp_words[i]:
                errors += weights[i] if i < len(weights) else 1.0
        errors += abs(len(ref_words) - len(hyp_words))

        reward = max(0.0, 1.0 - errors / total_w)
        rewards.append(reward)

    return torch.tensor(rewards, dtype=torch.float32)

In [11]:
# ---- REWARD FUNCTION 3: LLM Scorer ----

def compute_llm_reward(
    hypotheses: List[str],
    references: List[str],
    domain: str = "healthcare",
    use_mock: bool = True,
) -> torch.Tensor:
    """
    LLM-based reward — mock or real.
    UNCHANGED from HF version.
    """
    if use_mock:
        return _mock_llm_reward(hypotheses, references)
    else:
        return _real_llm_reward(hypotheses, references, domain)

In [12]:
def _mock_llm_reward(hypotheses, references):
    rewards = []
    for hyp, ref in zip(hypotheses, references):
        if not ref.strip():
            rewards.append(0.5)
            continue
        base  = max(0.0, 1.0 - compute_wer_jiwer(ref, hyp))
        noise = np.random.normal(0, 0.05)
        rewards.append(float(np.clip(base + noise, 0, 1)))
    return torch.tensor(rewards, dtype=torch.float32)

In [13]:
def _real_llm_reward(hypotheses, references, domain):
    import anthropic
    client = anthropic.Anthropic()

    SYSTEM = f"""You are a {domain} transcription quality evaluator.
Score the hypothesis vs reference from 0.0 to 1.0.
Pay special attention to {domain}-specific terminology.
Respond ONLY with JSON: {{"score": <float>}}"""

    rewards = []
    for hyp, ref in zip(hypotheses, references):
        try:
            resp   = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=50,
                system=SYSTEM,
                messages=[{"role": "user", "content": f"Reference: {ref}\nHypothesis: {hyp}"}],
            )
            result = json.loads(resp.content[0].text)
            rewards.append(float(result["score"]))
        except Exception as e:
            logger.warning(f"LLM reward failed: {e} — using 0.5")
            rewards.append(0.5)
    return torch.tensor(rewards, dtype=torch.float32)

# CELL 8 — NeMo Model Subclass with Reward Loss Hook
HIS IS THE KEY DIFFERENCE FROM HF VERSION.

 In HF: we subclassed Seq2SeqTrainer and overrode compute_loss()
 In NeMo: we subclass EncDecCTCModelBPE and override _compute_loss()

 NeMo's _compute_loss() receives:
   - log_probs     : CTC log probabilities (batch, time, vocab)
   - targets       : ground truth token ids
   - input_lengths : audio lengths
   - target_lengths: transcript lengths

 The CTC loss is computed here — this is where we inject reward weighting.


In [49]:
# def load_nemo_model_with_reward(model_name: str, reward_mode: str = "mwer", reward_weight: float = 0.3):
#     """
#     Load pretrained NeMo model and attach reward attributes.

#     For SFT baseline: reward injection is disabled (just CTC loss).
#     For RL stage: override training_step in a callback or subclass properly.
#     """
#     model = EncDecCTCModelBPE.from_pretrained(model_name)

#     # Attach reward config as attributes (for later use)
#     model.reward_mode   = reward_mode
#     model.reward_weight = reward_weight
#     model._step_logs    = []

#     logger.info(f"  Model loaded with reward config: mode={reward_mode}, weight={reward_weight}")
#     return model

In [60]:
def load_nemo_model_with_reward(model_name: str, reward_mode: str = "mwer", reward_weight: float = 0.3):
    """
    Load pretrained NeMo model with reward-augmented loss.

    For SFT baseline: set reward_weight=0.0
    For RL stage: set reward_weight=0.3 (or tune)
    """
    import types

    model = EncDecCTCModelBPE.from_pretrained(model_name)

    # Attach reward config as attributes
    model.reward_mode   = reward_mode
    model.reward_weight = reward_weight
    model._step_logs    = []

    # Store original training_step
    original_training_step = model.training_step

    def patched_training_step(self, batch, batch_nb):
        """
        Override training_step with optional reward injection.
        """
        # Call original training_step to get the standard result
        result = original_training_step(batch, batch_nb)

        # If reward_weight is 0, return standard result (SFT mode)
        if self.reward_weight == 0.0:
            return result

        # ---- REWARD INJECTION ----
        # Extract loss from result
        if isinstance(result, dict):
            ctc_loss = result['loss']
        else:
            ctc_loss = result

        # Get batch data for decoding
        # Handle different batch formats (Lhotse vs tuple)
        if hasattr(batch, 'audio'):
            # Lhotse batch format
            signal = batch.audio
            signal_len = batch.audio_lens
            transcript = batch.tokens
            transcript_len = batch.token_lens
        elif hasattr(batch, 'input_signal'):
            signal = batch.input_signal
            signal_len = batch.input_signal_length
            transcript = batch.targets
            transcript_len = batch.target_lengths
        else:
            # Tuple format: (signal, signal_len, transcript, transcript_len, ...)
            signal, signal_len, transcript, transcript_len = batch[:4]

        with torch.no_grad():
            # Forward pass to get log_probs
            log_probs, encoded_len, _ = self.forward(
                input_signal=signal, input_signal_length=signal_len
            )

            # Decode predictions to text using NeMo's decoding
            hyps_raw = self.wer.decoding.ctc_decoder_predictions_tensor(
                decoder_outputs=log_probs,
                decoder_lengths=encoded_len,
                return_hypotheses=False,
            )

            # Handle different return formats - extract text if Hypothesis objects
            hyps = []
            for h in hyps_raw:
                if isinstance(h, str):
                    hyps.append(h)
                elif hasattr(h, 'text'):
                    # Hypothesis object - extract text attribute
                    hyps.append(h.text)
                else:
                    # Fallback - convert to string
                    hyps.append(str(h))

            # Decode references from token IDs
            refs = []
            for i in range(transcript.size(0)):
                target_len = transcript_len[i].item()
                target_ids = transcript[i, :target_len].tolist()
                ref_text = self.tokenizer.ids_to_text(target_ids)
                refs.append(ref_text)

        # Compute reward based on mode
        if self.reward_mode == "mwer":
            rewards = compute_mwer_reward(hyps, refs).to(ctc_loss.device)
        elif self.reward_mode == "wwer":
            rewards = compute_wwer_reward(hyps, refs).to(ctc_loss.device)
        elif self.reward_mode == "llm":
            rewards = compute_llm_reward(hyps, refs, use_mock=True).to(ctc_loss.device)
        elif self.reward_mode == "all":
            r_mwer = compute_mwer_reward(hyps, refs).to(ctc_loss.device)
            r_wwer = compute_wwer_reward(hyps, refs).to(ctc_loss.device)
            r_llm  = compute_llm_reward(hyps, refs, use_mock=True).to(ctc_loss.device)
            rewards = (r_mwer + r_wwer + r_llm) / 3.0
        else:
            rewards = compute_mwer_reward(hyps, refs).to(ctc_loss.device)

        # Reward-augmented loss: penalize low-reward predictions
        penalty = 1.0 - rewards.mean()
        total_loss = ctc_loss + self.reward_weight * penalty

        # Log for debugging
        self._step_logs.append({
            "ctc_loss": ctc_loss.item(),
            "reward_mean": rewards.mean().item(),
            "penalty": penalty.item(),
            "total_loss": total_loss.item(),
            "reward_mode": self.reward_mode,
        })

        # Update result with new loss
        if isinstance(result, dict):
            result['loss'] = total_loss
            if 'log' in result:
                result['log']['reward_mean'] = rewards.mean()
                result['log']['penalty'] = penalty
        else:
            result = total_loss

        return result

    # Bind the patched method
    model.training_step = types.MethodType(patched_training_step, model)

    logger.info(f"  Model loaded with reward config: mode={reward_mode}, weight={reward_weight}")
    return model

In [55]:
class NemoRewardModel(EncDecCTCModelBPE):
    """
    NeMo Conformer-CTC with reward-based loss injection.

    Uses proper NeMo restoration flow to avoid tokenizer path issues.
    """

    def __init__(self, cfg: DictConfig, trainer=None,
                 reward_mode: str = "mwer", reward_weight: float = 0.3):
        super().__init__(cfg=cfg, trainer=trainer)
        self.reward_mode   = reward_mode
        self.reward_weight = reward_weight
        self._step_logs    = []

    @classmethod
    def from_pretrained_with_reward(
        cls, model_name: str, reward_mode: str = "mwer", reward_weight: float = 0.3
    ):
        """
        Load pretrained NeMo model with reward capability.

        FIX: Use NeMo's restore_from with override_config_path pattern,
        or simply add attributes to base model without class swap.
        """
        # Load base model properly through NeMo's restore mechanism
        base_model = EncDecCTCModelBPE.from_pretrained(model_name)

        # Instead of class swap, just add reward attributes directly
        # The base EncDecCTCModelBPE already inherits from LightningModule
        base_model.reward_mode   = reward_mode
        base_model.reward_weight = reward_weight
        base_model._step_logs    = []

        # Monkey-patch the _compute_loss method
        original_compute_loss = base_model._compute_loss

        def patched_compute_loss(log_probs, targets, input_lengths, target_lengths):
            # Standard CTC loss
            ctc_loss = base_model.loss(
                log_probs=log_probs,
                targets=targets,
                input_lengths=input_lengths,
                target_lengths=target_lengths,
            )

            # Log hook point passage
            base_model._step_logs.append({
                "ctc_loss":    ctc_loss.item(),
                "reward_mode": base_model.reward_mode,
            })

        # ---- REWARD INJECTION BLOCK (uncomment to activate) ----
        # with torch.no_grad():
        #     # Greedy decode current batch
        #     greedy_ids = torch.argmax(log_probs, dim=-1)   # (batch, T)
        #     hyps = self._wer.decoding.ctc_decoder_predictions_tensor(
        #         greedy_ids, predictions_len=input_lengths, return_hypotheses=False
        #     )
        #     refs = self.tokenizer.ids_to_text(targets)
        #
        # if self.reward_mode == "mwer":
        #     rewards = compute_mwer_reward(hyps, refs).to(ctc_loss.device)
        # elif self.reward_mode == "wwer":
        #     rewards = compute_wwer_reward(hyps, refs).to(ctc_loss.device)
        # elif self.reward_mode == "llm":
        #     rewards = compute_llm_reward(hyps, refs, use_mock=True).to(ctc_loss.device)
        #
        # penalty    = (1.0 - rewards.mean())
        # total_loss = ctc_loss + self.reward_weight * penalty
        # return total_loss
        # --------------------------------------------------------

            return ctc_loss

        # Bind the patched method
        import types
        base_model._compute_loss = types.MethodType(
            lambda self, *args, **kwargs: patched_compute_loss(*args, **kwargs),
            base_model
        )

        logger.info(f"  Reward model loaded: mode={reward_mode}, weight={reward_weight}")
        return base_model

# CELL 9 — Training Logger (PyTorch Lightning Callback)
 NeMo uses PL Callbacks instead of HF TrainerCallback.
 Functionally identical logging — just different base class.

In [32]:
class NemoTrainingLogger(Callback):
    """
    Documents NeMo training loop behaviour.
    PyTorch Lightning equivalent of the HF TrainerCallback.
    Logs: loss per step, WER per epoch, time, GPU memory.
    """

    def __init__(self):
        self.epoch_start = None
        self.step_logs   = []

    def on_train_epoch_start(self, trainer, pl_module):
        self.epoch_start = time.time()
        epoch = trainer.current_epoch + 1
        logger.info(f"\n{'='*50}")
        logger.info(f"EPOCH {epoch} STARTED")
        logger.info(f"{'='*50}")

    def on_train_epoch_end(self, trainer, pl_module):
        elapsed = time.time() - self.epoch_start
        mem_gb  = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
        epoch   = trainer.current_epoch
        logger.info(f"EPOCH {epoch} COMPLETE | Time: {elapsed:.1f}s | GPU mem: {mem_gb:.2f}GB")

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        step = trainer.global_step
        if step % 50 == 0:
            loss = outputs["loss"].item() if isinstance(outputs, dict) else float(outputs)
            lr   = trainer.optimizers[0].param_groups[0]["lr"]
            logger.info(f"  Step {step:4d} | loss={loss:.4f} | lr={lr:.2e}")
            self.step_logs.append({"step": step, "loss": loss, "lr": lr})

# CELL 10 — REWARD FUNCTION TEST (run before training)

In [33]:
def test_reward_functions():
    """
    Sanity-check all three reward functions.
    UNCHANGED from HF version — reward functions are framework-agnostic.
    Run this before training to confirm rewards work correctly.
    """
    print("\n" + "="*60)
    print("REWARD FUNCTION TESTS")
    print("="*60)

    test_cases = [
        ("hello world",           "hello world"),
        ("hello word",            "hello world"),
        ("the plaintiff filed",   "the plaintiff filed"),
        ("the plantiff filed",    "the plaintiff filed"),
        ("completely wrong text", "hello world"),
    ]

    hyps = [t[0] for t in test_cases]
    refs = [t[1] for t in test_cases]

    print("\n--- MWER Rewards ---")
    mwer_r = compute_mwer_reward(hyps, refs)
    for (h, r), reward in zip(test_cases, mwer_r):
        print(f"  hyp: '{h}' | ref: '{r}' => reward={reward:.3f}")

    print("\n--- WWER Rewards (domain terms weighted 3x) ---")
    wwer_r = compute_wwer_reward(hyps, refs)
    for (h, r), reward in zip(test_cases, wwer_r):
        print(f"  hyp: '{h}' | ref: '{r}' => reward={reward:.3f}")

    print("\n--- LLM Rewards (mock) ---")
    llm_r = compute_llm_reward(hyps, refs, use_mock=True)
    for (h, r), reward in zip(test_cases, llm_r):
        print(f"  hyp: '{h}' | ref: '{r}' => reward={reward:.3f}")

    print("\n[PASS] All reward functions returned tensors of correct shape")
    print("Note: WWER should penalise 'plantiff' harder than MWER — check above!")
    return mwer_r, wwer_r, llm_r

# CELL 11 — MAIN NEMO TRAINING PIPELINE


In [52]:
def run_nemo_baseline_training():
    """
    Full NeMo pipeline with proper NeMo 2.x trainer integration.
    """

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1. Load dataset
    logger.info("Step 1: Loading dataset ...")
    try:
        dataset = load_common_voice(TRAIN_SAMPLES, EVAL_SAMPLES)
        text_field = "sentence"
        logger.info("  Using Common Voice 17.0")
    except Exception as e:
        logger.warning(f"CV17 failed ({e}). Falling back to LibriSpeech.")
        dataset, text_field = load_librispeech_fallback(TRAIN_SAMPLES, EVAL_SAMPLES)

    # 2. Build NeMo manifests (save audio + write JSON)
    logger.info("Step 2: Building NeMo manifests ...")
    audio_dir      = os.path.join(OUTPUT_DIR, "audio")
    train_manifest = build_nemo_manifest(dataset["train"],      "train", audio_dir, text_field)
    val_manifest   = build_nemo_manifest(dataset["validation"], "val",   audio_dir, text_field)

    # 3. Load model
    logger.info("Step 3: Loading NeMo model ...")
    # model = load_nemo_model_with_reward(
    #     NEMO_MODEL_NAME,
    #     reward_mode=REWARD_MODE,
    #     reward_weight=0.3,
    # )

    model = load_nemo_model_with_reward(
        NEMO_MODEL_NAME,
        reward_mode=REWARD_MODE,
        reward_weight=0.3,  # Set to 0.0 for pure SFT, 0.3 for RL
    )

    # 4. Set datasets on model
    logger.info("Step 4: Configuring datasets ...")
    data_cfg = build_data_config(train_manifest, val_manifest)
    model.setup_training_data(data_cfg.train_ds)
    model.setup_validation_data(data_cfg.validation_ds)

    # 5. Create trainer config for NeMo 2.x
    logger.info("Step 5: Setting up NeMo trainer ...")

    # Use exp_manager for proper NeMo training setup
    trainer_config = OmegaConf.create({
        "trainer": {
            "devices": 1,
            "accelerator": "gpu" if torch.cuda.is_available() else "cpu",
            "max_epochs": NUM_EPOCHS,
            "precision": "16-mixed",  # NeMo 2.x syntax for fp16
            "log_every_n_steps": 25,
            "val_check_interval": 1.0,
            "enable_checkpointing": False,  # We'll handle this manually
            "logger": False,  # Disable default logger
        }
    })

    # Create trainer using NeMo's expected pattern
    trainer = pl.Trainer(**trainer_config.trainer)

    # CRITICAL: Set trainer on model BEFORE setup_optimization
    model.set_trainer(trainer)

    # 6. Configure optimizer (AFTER setting trainer)
    logger.info("Step 6: Configuring optimizer ...")
    model.setup_optimization(OmegaConf.create({
        "name": "adamw",
        "lr": LEARNING_RATE,
        "weight_decay": 1e-3,
        "sched": {
            "name": "CosineAnnealing",
            "warmup_steps": 100,
            "min_lr": 1e-6,
        },
    }))

    # 7. Train
    logger.info("Step 7: Starting NeMo training ...")
    start = time.time()
    trainer.fit(model)
    elapsed = time.time() - start

    logger.info(f"\nTraining complete! Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

    # 8. Save model
    save_path = os.path.join(OUTPUT_DIR, "nemo_model.nemo")
    model.save_to(save_path)
    logger.info(f"Model saved: {save_path}")

    # 9. Evaluate WER
    logger.info("Step 8: Evaluating WER ...")
    final_wer = evaluate_wer(model, val_manifest)
    logger.info(f"Final WER: {final_wer:.2f}%")

    return model, final_wer

# CELL 12 — RUN EVERYTHING


In [45]:
# DIAGNOSTIC: Check model inheritance
model = EncDecCTCModelBPE.from_pretrained(NEMO_MODEL_NAME)
print(f"Model type: {type(model)}")
print(f"Model MRO: {type(model).__mro__}")
print(f"Is LightningModule: {isinstance(model, pl.LightningModule)}")

import nemo.core
print(f"Is NeMo ModelPT: {isinstance(model, nemo.core.ModelPT)}")

[NeMo I 2026-03-31 17:58:18 cloud:58] Found existing object /root/.cache/torch/NeMo/NeMo_2.7.2/stt_en_conformer_ctc_small/5d2d8e5b2b5adb8f5091363c6ba19c55/stt_en_conformer_ctc_small.nemo.
[NeMo I 2026-03-31 17:58:18 cloud:64] Re-using file from: /root/.cache/torch/NeMo/NeMo_2.7.2/stt_en_conformer_ctc_small/5d2d8e5b2b5adb8f5091363c6ba19c55/stt_en_conformer_ctc_small.nemo
[NeMo I 2026-03-31 17:58:18 common:939] Instantiating model from pre-trained checkpoint
[NeMo I 2026-03-31 17:58:19 mixins:184] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-03-31 17:58:20 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /data/NeMo_ASR_SET/English/v2.0/train/tarred_audio_manifest.json
    sample_rate: 16000
    batch_size: 64
    shuffle: true
    num_workers: 8
    pin_memory: true
    use_start_end_token: false
    trim_silence: false
    max_duration: 20.0
    min_duration: 0.1
    shuffle_n: 2048
    is_tarred: true
    tarred_audio_filepaths: /data/NeMo_ASR_SET/English/v2.0/train/audio__OP_0..4095_CL_.tar
    
[NeMo W 2026-03-31 17:58:20 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath:
    - /data/ASR/LibriSpeech/librispeech_withs

[NeMo I 2026-03-31 17:58:20 save_restore_connector:285] Model EncDecCTCModelBPE was successfully restored from /root/.cache/torch/NeMo/NeMo_2.7.2/stt_en_conformer_ctc_small/5d2d8e5b2b5adb8f5091363c6ba19c55/stt_en_conformer_ctc_small.nemo.
Model type: <class 'nemo.collections.asr.models.ctc_bpe_models.EncDecCTCModelBPE'>
Model MRO: (<class 'nemo.collections.asr.models.ctc_bpe_models.EncDecCTCModelBPE'>, <class 'nemo.collections.asr.models.ctc_models.EncDecCTCModel'>, <class 'nemo.collections.asr.models.asr_model.ASRModel'>, <class 'nemo.core.classes.modelPT.ModelPT'>, <class 'lightning.pytorch.core.module.LightningModule'>, <class 'lightning.fabric.utilities.device_dtype_mixin._DeviceDtypeModuleMixin'>, <class 'lightning.pytorch.core.mixins.hparams_mixin.HyperparametersMixin'>, <class 'lightning.pytorch.core.hooks.ModelHooks'>, <class 'lightning.pytorch.core.hooks.DataHooks'>, <class 'lightning.pytorch.core.hooks.CheckpointHooks'>, <class 'torch.nn.modules.module.Module'>, <class 'nemo.

In [46]:
if __name__ == "__main__":

    print("\n" + "="*60)
    print("STEP 1: Testing reward functions ...")
    print("="*60)
    test_reward_functions()

    print("\n" + "="*60)
    print("STEP 2: Running NeMo baseline training ...")
    print("="*60)
    model, wer = run_nemo_baseline_training()

    print("\n" + "="*60)
    print(f"DONE. Final WER: {wer:.2f}%")
    print("Next: uncomment reward injection in NemoRewardModel._compute_loss()")
    print("="*60)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'mozilla-foundation/common_voice_17_0' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'mozilla-foundation/common_voice_17_0' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'librispeech_asr' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datas


STEP 1: Testing reward functions ...

REWARD FUNCTION TESTS

--- MWER Rewards ---
  hyp: 'hello world' | ref: 'hello world' => reward=1.000
  hyp: 'hello word' | ref: 'hello world' => reward=0.500
  hyp: 'the plaintiff filed' | ref: 'the plaintiff filed' => reward=1.000
  hyp: 'the plantiff filed' | ref: 'the plaintiff filed' => reward=0.667
  hyp: 'completely wrong text' | ref: 'hello world' => reward=0.000

--- WWER Rewards (domain terms weighted 3x) ---
  hyp: 'hello world' | ref: 'hello world' => reward=1.000
  hyp: 'hello word' | ref: 'hello world' => reward=0.500
  hyp: 'the plaintiff filed' | ref: 'the plaintiff filed' => reward=1.000
  hyp: 'the plantiff filed' | ref: 'the plaintiff filed' => reward=0.400
  hyp: 'completely wrong text' | ref: 'hello world' => reward=0.000

--- LLM Rewards (mock) ---
  hyp: 'hello world' | ref: 'hello world' => reward=1.000
  hyp: 'hello word' | ref: 'hello world' => reward=0.520
  hyp: 'the plaintiff filed' | ref: 'the plaintiff filed' => rewa

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

[NeMo I 2026-03-31 17:58:30 cloud:58] Found existing object /root/.cache/torch/NeMo/NeMo_2.7.2/stt_en_conformer_ctc_small/5d2d8e5b2b5adb8f5091363c6ba19c55/stt_en_conformer_ctc_small.nemo.
[NeMo I 2026-03-31 17:58:30 cloud:64] Re-using file from: /root/.cache/torch/NeMo/NeMo_2.7.2/stt_en_conformer_ctc_small/5d2d8e5b2b5adb8f5091363c6ba19c55/stt_en_conformer_ctc_small.nemo
[NeMo I 2026-03-31 17:58:30 common:939] Instantiating model from pre-trained checkpoint
[NeMo I 2026-03-31 17:58:31 mixins:184] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


...

[NeMo I 2026-03-31 17:58:42 wer:336] 
    
[NeMo I 2026-03-31 17:58:42 wer:337] WER reference:and if he does it will not be the reward of a hundred dollars for information that will make him tell on the second day about noon some of the boys were busy near the cabin laying in an extra supply of firewood
[NeMo I 2026-03-31 17:58:42 wer:338] WER predicted:and if he does it will not be the reward of a hundred dollars for information that will make and tell on the second day about noon some of the boys were busy near the town laying in an extra supply a firewod


Validation: |          | 0/? [00:00<?, ?it/s]

[NeMo I 2026-03-31 17:58:42 wer:336] 
    
[NeMo I 2026-03-31 17:58:42 wer:337] WER reference:he was in a fevered state of mind owing to the blight his wife's action threatened to cast upon his entire future
[NeMo I 2026-03-31 17:58:42 wer:338] WER predicted:he was in a fevered state of mind owing to the blight his wife's action threatened to cast upon his entire future
[NeMo I 2026-03-31 17:58:42 wer:336] 
    
[NeMo I 2026-03-31 17:58:42 wer:337] WER reference:for some reason he felt as if something might come that way and was relieved when all the envelopes had been scanned and nothing suspicious noticed
[NeMo I 2026-03-31 17:58:42 wer:338] WER predicted:some reason he felt if something might come that way and was relieved when all the envelopes had been scanned and nothing suspicious noticed
[NeMo I 2026-03-31 17:58:43 wer:336] 
    
[NeMo I 2026-03-31 17:58:43 wer:337] WER reference:fortunately there was nothing from his wife either
[NeMo I 2026-03-31 17:58:43 wer:338] WER predict

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=1` reached.
[NeMo W 2026-03-31 17:58:44 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-03-31 17:58:44 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 13it [00:05,  2.32it/s]


DONE. Final WER: 8.63%
Next: uncomment reward injection in NemoRewardModel._compute_loss()


RL (Reward Enabled)

In [61]:
if __name__ == "__main__":

    print("\n" + "="*60)
    print("STEP 1: Testing reward functions ...")
    print("="*60)
    test_reward_functions()

    print("\n" + "="*60)
    print("STEP 2: Running NeMo baseline training ...")
    print("="*60)
    model, wer = run_nemo_baseline_training()

    print("\n" + "="*60)
    print(f"DONE. Final WER: {wer:.2f}%")
    print("Next: uncomment reward injection in NemoRewardModel._compute_loss()")
    print("="*60)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'mozilla-foundation/common_voice_17_0' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'mozilla-foundation/common_voice_17_0' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'librispeech_asr' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datas


STEP 1: Testing reward functions ...

REWARD FUNCTION TESTS

--- MWER Rewards ---
  hyp: 'hello world' | ref: 'hello world' => reward=1.000
  hyp: 'hello word' | ref: 'hello world' => reward=0.500
  hyp: 'the plaintiff filed' | ref: 'the plaintiff filed' => reward=1.000
  hyp: 'the plantiff filed' | ref: 'the plaintiff filed' => reward=0.667
  hyp: 'completely wrong text' | ref: 'hello world' => reward=0.000

--- WWER Rewards (domain terms weighted 3x) ---
  hyp: 'hello world' | ref: 'hello world' => reward=1.000
  hyp: 'hello word' | ref: 'hello world' => reward=0.500
  hyp: 'the plaintiff filed' | ref: 'the plaintiff filed' => reward=1.000
  hyp: 'the plantiff filed' | ref: 'the plaintiff filed' => reward=0.400
  hyp: 'completely wrong text' | ref: 'hello world' => reward=0.000

--- LLM Rewards (mock) ---
  hyp: 'hello world' | ref: 'hello world' => reward=1.000
  hyp: 'hello word' | ref: 'hello world' => reward=0.575
  hyp: 'the plaintiff filed' | ref: 'the plaintiff filed' => rewa

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

[NeMo I 2026-03-31 18:23:02 cloud:58] Found existing object /root/.cache/torch/NeMo/NeMo_2.7.2/stt_en_conformer_ctc_small/5d2d8e5b2b5adb8f5091363c6ba19c55/stt_en_conformer_ctc_small.nemo.
[NeMo I 2026-03-31 18:23:02 cloud:64] Re-using file from: /root/.cache/torch/NeMo/NeMo_2.7.2/stt_en_conformer_ctc_small/5d2d8e5b2b5adb8f5091363c6ba19c55/stt_en_conformer_ctc_small.nemo
[NeMo I 2026-03-31 18:23:02 common:939] Instantiating model from pre-trained checkpoint
[NeMo I 2026-03-31 18:23:03 mixins:184] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


...

[NeMo I 2026-03-31 18:23:14 wer:336] 
    
[NeMo I 2026-03-31 18:23:14 wer:337] WER reference:i flung myself into this rapid noisy and volcanic life which had formerly terrified me when i thought of it and which had become for me the necessary complement of my love for marguerite what else could i have done
[NeMo I 2026-03-31 18:23:14 wer:338] WER predicted:i flung myself this rapid noisy andlcanic life which formerlyerrified me of it and which had become for me the necessary complement of my love for could i have go


Validation: |          | 0/? [00:00<?, ?it/s]

[NeMo I 2026-03-31 18:23:14 wer:336] 
    
[NeMo I 2026-03-31 18:23:14 wer:337] WER reference:he was in a fevered state of mind owing to the blight his wife's action threatened to cast upon his entire future
[NeMo I 2026-03-31 18:23:14 wer:338] WER predicted:he was in a fevered state of mind owing to the blight his wife's action threatened to cast upon his entire future
[NeMo I 2026-03-31 18:23:14 wer:336] 
    
[NeMo I 2026-03-31 18:23:14 wer:337] WER reference:for some reason he felt as if something might come that way and was relieved when all the envelopes had been scanned and nothing suspicious noticed
[NeMo I 2026-03-31 18:23:14 wer:338] WER predicted:some reason he felt if something might come that way and was red when all the envelopes had been scanned and nothing suspicious noticed
[NeMo I 2026-03-31 18:23:15 wer:336] 
    
[NeMo I 2026-03-31 18:23:15 wer:337] WER reference:fortunately there was nothing from his wife either
[NeMo I 2026-03-31 18:23:15 wer:338] WER predicted:fo

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=1` reached.
[NeMo W 2026-03-31 18:23:16 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-03-31 18:23:16 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 0it [00:00, ?it/s]
Transcribing: 1it [00:00,  3.06it/s]
Transcribing: 2it [00:00,  3.18it/s]
Transcribing: 3it [00:00,  3.67it/s]
Transcribing: 4it [00:01,  3.66it/s]
Transcribing: 5it [00:01,  2.75it/s]
Transcribing: 6it [00:02,  2.22it/s]
Transcribing: 7it [00:02,  2.21it/s]
Transcribing: 8it [00:03,  1.92it/s]
Transcribing:


DONE. Final WER: 12.12%
Next: uncomment reward injection in NemoRewardModel._compute_loss()
